# Module 02 — Write Problems (Colab)

NovaBridge's agent charges a client's **$2,500 quarterly advisory fee**. Agents retry when something times out. So: what happens to the client's card when the agent runs twice?

You will run the agent yourself, step by step, watch it charge the client **twice**, see why a database constraint does **not** save you, and fix it.

> ▶️ **Run every cell in order, top to bottom.** If a cell errors, re-run the setup cell first (it is idempotent), then continue.

## 1. Set up (Postgres + recorded model outputs, ~2 min)

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
!SKIP_OLLAMA=1 bash setup.sh

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'
!python preflight.py

## 2. Meet the agent

The "agent" is just code you can call. Let's wire it up and look at what it reads: the client's billing note.

In [ ]:
import sys, importlib
sys.path.insert(0, 'modules/02_write_path')

from nova.llm import get_llm
from nova.store import get_store
from nova.effects import PaymentGateway
from nova.agent import payment_memo_prompt, load_document
import your_fix

llm = get_llm()                 # the model (answers replayed from recordings)
store = get_store(); store.init_schema(); store.reset_demo()
gateway = PaymentGateway()      # the payment processor (the outside world)

doc = load_document('alpha', 'billing_instruction.md')
print(doc)                      # <-- what the agent reads

The agent reads that note and writes a one-line **payment memo**. Run it:

In [ ]:
memo_1 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 1))
print(memo_1)

Agents **retry** when something fails. Run the agent again — and watch the wording change, even though it's the same fee:

In [ ]:
memo_2 = llm.complete(payment_memo_prompt('alpha', 'Q1-2026', doc, 2))
print('Run 1:', memo_1)
print('Run 2:', memo_2)
print('Same text?', memo_1 == memo_2)   # False: same charge, different words

## 3. The naive way: charge, then record

Here is the actual code that charges the fee. Read it — this is the file you'll fix.

In [ ]:
print(open('modules/02_write_path/your_fix.py').read())

It charges the gateway, then records a row in the `charges` table keyed on `(client, period)` — which is **unique**, so the table can only hold one row per fee.

Now run the agent **and its retry** through it, then look at two things: your database, and what the gateway actually did.

In [ ]:
store.reset_demo(); gateway = PaymentGateway()

for memo in (memo_1, memo_2):          # the agent runs, then retries
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo)

print('Your charges table :', store.get_charges('alpha'))        # ONE row
print('Gateway ledger     :', gateway.charges_for('alpha'))      # TWO charges
print()
print(f"Your DB says you charged {len(store.get_charges('alpha'))} time.")
print(f"The gateway charged the client {len(gateway.charges_for('alpha'))} times = ${gateway.total_charged('alpha')}.")

See it in the **real database** — one clean row:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, amount FROM charges;"

### The aha

Your database looks perfect: **one** charge. But the payment gateway charged the client **twice** — **$5,000** left the card.

The unique key on your `charges` table made your **record** idempotent. It did **not** un-charge the card. The gateway is the outside world; the money moved the instant you called `charge()`, *before* you recorded anything. **A database constraint protects your data, not the client's card.**

So the fix isn't a better constraint on the record. It's to **stop the charge before it happens**, keyed on the agent's stable intent `(client, period)` — not on the memo text, which changes every retry.

## 4. Fix it

Edit the cell below: **before** calling `gateway.charge(...)`, check `store.already_charged(key)` and `return` if this fee was already charged. Then run the cell to save it.

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def charge_key(client_id, billing_period):
    """Stable key for one fee: the agent's intent (client + period)."""
    return hashlib.sha256(f"{client_id}|{billing_period}".encode("utf-8")).hexdigest()


def charge_client_fee(gateway, store, client_id, billing_period, amount, memo):
    key = charge_key(client_id, billing_period)
    # TODO: if store.already_charged(key): return   <-- guard the charge here
    gateway.charge(client_id, amount, memo)
    store.record_charge(key, client_id, amount)

Reload your fixed code and run the retry again. Watch `already_charged` flip to `True` on the second attempt, and the gateway stay at **one** charge:

In [ ]:
importlib.reload(your_fix)
store.reset_demo(); gateway = PaymentGateway()
key = your_fix.charge_key('alpha', 'Q1-2026')

for i, memo in enumerate((memo_1, memo_2), start=1):
    seen_before = store.already_charged(key)
    your_fix.charge_client_fee(gateway, store, 'alpha', 'Q1-2026', 2500, memo)
    print(f'attempt {i}: already_charged before = {seen_before}  ->  gateway now has {len(gateway.charges_for("alpha"))} charge(s)')

print()
print(f"The gateway charged the client {len(gateway.charges_for('alpha'))} time(s) = ${gateway.total_charged('alpha')}.")

The second attempt saw `already_charged = True`, so it never touched the gateway. The client is charged once. **That** is idempotency at the effect boundary.

## 5. Prove it

In [ ]:
!python -m pytest modules/02_write_path/test_write.py -v

### Optional: run the real local model

Everything above replayed the model's answers from recordings. To generate them live with the real open-source model (slower — installs Ollama and pulls a ~1.3 GB model), run `!bash setup.sh`, set `os.environ['NOVA_LLM']='ollama'`, and re-run from section 2.